In [11]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from ranking_methods import rank_accuracy
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies, rank_accuracy
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
import json
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [12]:
best_feat = 'ibi_median'
#data_folder = "./data/dados_2026_06_08"
data_folder = "./data/dados_2026_05_01"
#data_folder = "./data/dados_iniciais_estruturados"
dataFiles = glob.glob(f'{data_folder}/*.zip')
output_folder = './results/predict_to_predict2'
ground_file = f'{data_folder}/ground.json'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

if os.path.exists(ground_file):
    with open(ground_file, 'r') as f:
        ranks_ground = json.load(f)
        #ranks_ground = [int(f.split("_")[1]) for f in list(ranks_ground.keys())]

for k, v in ranks_ground.items():
    print(f"{k}: {v}")
root_folder = f"{data_folder}"    



['./data/dados_2026_05_01/2026-04-16 17.41.32.zip', './data/dados_2026_05_01/2026-04-23 17.58.52.zip', './data/dados_2026_05_01/2026-04-14 17.19.07.zip', './data/dados_2026_05_01/2026-04-24 17.31.38.zip', './data/dados_2026_05_01/2026-04-22 10.43.23.zip', './data/dados_2026_05_01/2026-04-24 19.33.09.zip', './data/dados_2026_05_01/2026-04-17 19.18.35.zip']
animal_6: 7
animal_10: 8
animal_11: 5
animal_9: 4
animal_1: 6
animal_7: 2
animal_2: 10
animal_12: 3
animal_4: 9
animal_8: 12
animal_5: 1
animal_3: 11


In [3]:

for file in dataFiles:
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)
    a = chr.intellicage_unwrapper([file], sub_folder, sampling_interval = '30T')



File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_1.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_10.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_11.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_12.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_2.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_3.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_4.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_5.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_6.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_7.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_8.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_9.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_1.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_10.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_11.txt
File saved in ./data

In [13]:
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)
individual_files = [f for f in individual_files if "animal_" in f]
apply_filtering = True
animals = [int(k.split("_")[1]) for k in ranks_ground.keys()]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)

Animal 1: 7
Animal 2: 7
Animal 3: 7
Animal 4: 7
Animal 5: 7
Animal 6: 7
Animal 7: 7
Animal 8: 7
Animal 9: 7
Animal 10: 7
Animal 11: 7
Animal 12: 7
Animal2 1
savgol True
Animal2 2
savgol True
Animal2 3
savgol True
Animal2 4
savgol True
Animal2 5
savgol True
Animal2 6
savgol True
Animal2 7
savgol True
Animal2 8
savgol True
Animal2 9
savgol True
Animal2 10
savgol True
Animal2 11
savgol True
Animal2 12
savgol True
Animal animal_1 Days found: 15
Animal animal_2 Days found: 15
Animal animal_3 Days found: 15
Animal animal_4 Days found: 15
Animal animal_5 Days found: 15
Animal animal_6 Days found: 15
Animal animal_7 Days found: 15
Animal animal_8 Days found: 15
Animal animal_9 Days found: 15
Animal animal_10 Days found: 15
Animal animal_11 Days found: 15
Animal animal_12 Days found: 15


In [ ]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)
y = []
for animal in all_features['animal'].tolist():
    y.append(ranks_ground[animal])



for animal, rank in sorted(ranks_ground.items(), key=lambda x: x[1]):
    print(f"{rank}: {animal}")

all_features['actual_rank'] = y

# [9, 8, 11, 10, 12, 1, 5, 6, 7, 2, 3, 4]

all_features.head()

Saving features on ./results/predict_to_predict2/basic_features.csv
Saving temporal features on ./results/predict_to_predict2/temporal_features.csv
Saving all features on ./results/predict_to_predict2/all_features.csv
1: animal_5
2: animal_7
3: animal_12
4: animal_9
5: animal_11
6: animal_1
7: animal_6
8: animal_10
9: animal_4
10: animal_2
11: animal_3
12: animal_8


,animal,total_activity,mean_activity,std_activity,max_activity,min_activity,cv_activity,median_activity,max_median_ratio,activity_per_hour,...,night_bout_len_mean,night_bout_len_cv,night_transitions_per_hour,night_onset_latency_h,night_first_2h_frac,night_gini,night_peak_hour,night_activity_per_bout,night_day_intensity_ratio,actual_rank
animal_5,animal_5,2039.0,1.518243,1.973609,11.314286,-1.085714,1.299930,0.685714,16.500000,84.958333,...,0.575342,0.677317,0.870343,9.75,-0.061350,4.716969,1.0,0.920382,0.064747,1
animal_7,animal_7,2300.0,1.712584,2.101365,12.685714,-1.971429,1.227015,1.028571,12.333333,95.833333,...,0.512195,0.746496,0.977645,5.25,0.000000,0.000000,1.0,-0.056739,-0.003557,2
animal_12,animal_12,2009.0,1.495905,2.108541,11.657143,-1.457143,1.409542,0.685714,17.000000,83.708333,...,0.531646,0.786030,0.941878,5.25,-0.156146,11.069784,1.0,0.304689,0.016535,3
animal_9,animal_9,2205.0,1.641847,1.957898,12.000000,-2.228571,1.192498,1.028571,11.666667,91.875000,...,0.466667,0.649764,1.073025,5.25,0.000000,0.000000,1.0,-0.027003,-0.001879,4
animal_11,animal_11,1970.0,1.466865,2.035308,12.342857,-1.885714,1.387522,0.685714,18.000000,82.083333,...,0.552632,0.749527,0.906110,8.75,-0.048471,3.843520,1.0,1.092010,0.072007,5


In [19]:
feature_rhos_path = "./data/dados_iniciais_estruturados/feature_rhos.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [20]:

ys = [int(i.split("_")[1]) for i in all_features['animal'].tolist()]

best_feat = 'cosinor_amplitude'
combos = [['cosinor_amplitude'], ['power_24h', 'high_activity_frac', 'activity_per_bout']]



result = {}
cont = 0
for named_combo in combos:

    feature_rhos, proxies_raw = build_all_proxies(
        all_features, feature_cols, X_scaled, None,
        k=3, best_feat_idx=best_feat,
        named_combo=named_combo,
        feature_rhos=feature_rhos
    )
    for k, v in proxies_raw.items():
        print(f"{k}: {v}")



    best_feature_key = f'Best feature ({best_feat})'

    scores = proxies_raw[best_feature_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)


    output_best_feature = f'{data_folder}/pred_{best_feat}.csv'

    best_combo_key = f'Best combo ({ " + ".join(named_combo) })'
    scores = proxies_raw[best_combo_key]
    pred_rank = rankdata(scores, method='ordinal')


    combo_name = "_".join(named_combo)

    if combo_name not in result:
        summary = {"pred": pred_rank, "ground": ys}
        metrics = rank_accuracy(y, pred_rank)

        summary.update(metrics)

        result[combo_name] = summary


df = pd.DataFrame(result)
df.to_csv(f"./pred_data_to_predict2.csv", index=False)


print(result)


Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.42746586 0.26038611 0.26916381 0.3380771  0.37392806 0.24117553
 0.38022122 0.29245193 0.44404151 0.2809962  0.47956141 0.40175124]
Best combo (cosinor_amplitude): [-1.40897792 -0.88911819  1.70315471  1.23944159  1.02304615  0.40626647
 -1.15818333  0.68734123 -0.14392596 -0.7395635   0.32410904 -1.04359028]
Combo sign-aligned mean (cosinor_amplitude): [-1.40897792 -0.88911819  1.70315471  1.23944159  1.02304615  0.40626647
 -1.15818333  0.68734123 -0.14392596 -0.7395635   0.32410904 -1.04359028]
Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.42746586 0.26038611 0.26916381 0.3380771  0.37392806 0.24117553
 0.38022122 0.29245193 0.44404151 0.2809962  0.47956141 0.40175124]
Best combo (power_24h + high_activity_frac + activity_per_bout): [-2.60449622  1.5078055   2.93185325  0.54446378 -0.17062289 -0.17918256
 -1.54022